# 03 · Cleaning

### What this notebook does
**Interactions**
1. Drops rows with missing user, item, rating or timestamp, and ratings outside 1–5.
2. Converts millisecond timestamps and drops impossible dates.
3. Removes exact duplicates, then applies the **repeat policy** (keep the earliest purchase of each user–item pair).

**Metadata**
4. Removes duplicate products, strips HTML and boilerplate, normalises whitespace.
5. Normalises brand (`store`, with a fallback to `details["Brand"]`) and the category path.
6. Parses price to a number, truncates very long descriptions.

**Item set**
7. Drops items without a usable title (configurable), so all five models share exactly one item set.

Every step is logged with row counts in `interim/cleaning_log.json`.

### Why
A sequential model learns from *order*: duplicates, bad timestamps and ambiguous repeats corrupt the sequences it
learns from. Messy text produces poor embeddings for Model 1 and gives the LLM more room to hallucinate.

**Inputs:** `raw/ratings_5core.parquet`, `raw/meta_5core_items.parquet`
**Outputs:** `interim/ratings_clean.parquet`, `interim/meta_clean.parquet`, `interim/cleaning_log.json`

In [ ]:
# ---------------- Shared setup (identical in every notebook) ----------------
# All notebooks read and write inside one project folder (BASE).
# Colab: BASE lives in your Google Drive so outputs survive runtime restarts.
# Local Jupyter: BASE is ./298A_recsys next to the notebooks.
# Set the environment variable RECSYS_BASE to use any other folder.
import os, sys, json, re, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)

IN_COLAB = 'google.colab' in sys.modules
if os.environ.get('RECSYS_BASE'):
    BASE = Path(os.environ['RECSYS_BASE'])
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/298A_recsys')
else:
    BASE = Path('./298A_recsys').resolve()

RAW, INTERIM, PROCESSED = BASE / 'raw', BASE / 'interim', BASE / 'processed'
REPORTS, FIGS = BASE / 'reports', BASE / 'reports' / 'figures'
# Large downloads go to fast local disk in Colab (not Drive); re-downloaded if the runtime resets.
DOWNLOADS = Path(os.environ.get('RECSYS_DOWNLOAD_DIR', '/content/downloads' if IN_COLAB else str(BASE / 'downloads')))
for d in [RAW, INTERIM, PROCESSED, REPORTS, FIGS, DOWNLOADS]:
    d.mkdir(parents=True, exist_ok=True)

# Plot style: one hue per single-series chart, light recessive grid, no chart junk.
PRIMARY, ACCENT, MUTED = '#2a78d6', '#eb6834', '#8a8984'
plt.rcParams.update({
    'figure.figsize': (8, 4), 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': '#b5b4ae',
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'xtick.labelsize': 8,
    'ytick.labelsize': 8, 'axes.prop_cycle': matplotlib.cycler(color=[PRIMARY, ACCENT, '#1baf7a', '#eda100']),
})

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGS / f'{name}.png', bbox_inches='tight')
    plt.show()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def load_config():
    cfg_path = BASE / 'config.json'
    if not cfg_path.exists():
        raise FileNotFoundError('config.json not found - run 01_data_acquisition.ipynb first.')
    return json.loads(cfg_path.read_text())

print('Project folder:', BASE)

In [ ]:
import html

CFG = load_config()
ratings = pd.read_parquet(RAW / 'ratings_5core.parquet')
meta = pd.read_parquet(RAW / 'meta_5core_items.parquet')

LOG = []
def log_step(step, before, after, note=''):
    LOG.append({'step': step, 'rows_before': int(before), 'rows_after': int(after),
                'removed': int(before - after), 'note': note})
    print(f'{step:45s} {before:>11,} -> {after:>11,}  (-{before - after:,}) {note}')

## Part A · Interactions
### A1. Missing values and invalid ratings

In [ ]:
n0 = len(ratings)
ratings = ratings.dropna(subset=['user_id', 'parent_asin', 'rating', 'timestamp'])
ratings = ratings[(ratings.user_id.str.strip() != '') & (ratings.parent_asin.str.strip() != '')]
log_step('drop missing ids / rating / timestamp', n0, len(ratings))

n0 = len(ratings)
ratings = ratings[ratings.rating.between(1, 5)]
log_step('drop ratings outside 1-5', n0, len(ratings))

### A2. Timestamps
Amazon Reviews 2023 timestamps are Unix **milliseconds** (13 digits). We detect the unit rather than assume it,
then drop dates before Amazon's review system existed or after the dataset was collected.

In [ ]:
unit = 'ms' if ratings.timestamp.median() > 1e11 else 's'
ratings['dt'] = pd.to_datetime(ratings.timestamp, unit=unit, utc=True)
if unit == 's':
    ratings['timestamp'] = ratings['timestamp'] * 1000  # store everything in ms from here on
lo, hi = pd.Timestamp('1996-01-01', tz='UTC'), pd.Timestamp('2024-01-01', tz='UTC')
n0 = len(ratings)
ratings = ratings[(ratings.dt >= lo) & (ratings.dt < hi)]
log_step('drop timestamps outside 1996-2023', n0, len(ratings), f'detected unit = {unit}')

### A3. Duplicates and repeat purchases
- **Exact duplicates** (same user, item, rating, timestamp) are data errors → drop.
- **Repeat purchases** (same user and item at different times): we keep only the **earliest**. Reasons: the task is
  predicting the *next new* product, evaluation masks items the user already has, and SASRec benchmarks use the
  same convention. The number removed is logged so the choice can be reported.

In [ ]:
n0 = len(ratings)
ratings = ratings.drop_duplicates(['user_id', 'parent_asin', 'rating', 'timestamp'])
log_step('drop exact duplicate rows', n0, len(ratings))

assert CFG['repeat_policy'] == 'keep_first'
n0 = len(ratings)
ratings = (ratings.sort_values(['user_id', 'parent_asin', 'timestamp', 'row_id'])
                  .drop_duplicates(['user_id', 'parent_asin'], keep='first'))
log_step('repeat purchases: keep earliest per user-item', n0, len(ratings))

## Part B · Metadata
### B1. Duplicate products
One row per `parent_asin`. If duplicates exist we keep the row with the most information (longest text).

In [ ]:
def is_seq(v):
    return isinstance(v, (list, tuple, np.ndarray))

def to_list(v):
    return [str(x) for x in v] if is_seq(v) else []

for col in ['features', 'description', 'categories']:
    meta[col] = meta[col].apply(to_list)

meta['_info'] = (meta.title.fillna('').astype(str).str.len() + meta.description.apply(lambda l: sum(map(len, l)))
                 + meta.features.apply(lambda l: sum(map(len, l))))
n0 = len(meta)
meta = meta.sort_values('_info', ascending=False).drop_duplicates('parent_asin').drop(columns='_info')
log_step('metadata: drop duplicate parent_asin', n0, len(meta))

### B2. Text cleaning
- unescape HTML entities and remove tags (`<br>`, `<p>`, ...)
- remove common boilerplate ("Product Description", "From the manufacturer", "See more", ...)
- collapse whitespace; drop empty strings and exact repeated sentences within a field
- truncate descriptions to `max_desc_words` words (very long descriptions add noise and cost to LLM prompts)

In [ ]:
TAG_RE = re.compile(r'<[^>]+>')
URL_RE = re.compile(r'https?://\S+|www\.\S+')
WS_RE = re.compile(r'\s+')
BOILERPLATE_RE = re.compile(
    r'^(product description|from the manufacturer|about this item|see more|read more|'
    r'click here.*|important information|legal disclaimer.*)[:\s]*$', re.I)
JUNK_TITLE_RE = re.compile(r'^(n/?a|none|null|unknown|-+|\.+)$', re.I)

def is_missing(v):
    return v is None or (not is_seq(v) and pd.isna(v))

def clean_text(s):
    if is_missing(s):
        return ''
    s = html.unescape(str(s))
    s = TAG_RE.sub(' ', s)
    s = URL_RE.sub(' ', s)
    s = s.replace('​', ' ').replace('\xa0', ' ')
    return WS_RE.sub(' ', s).strip()

def clean_list(lst):
    out, seen = [], set()
    for x in lst:
        c = clean_text(x)
        if not c or BOILERPLATE_RE.match(c) or c.lower() in seen:
            continue
        seen.add(c.lower())
        out.append(c)
    return out

def truncate_words(s, n):
    w = s.split()
    return s if len(w) <= n else ' '.join(w[:n]) + ' ...'

before_html = meta.description.apply(lambda l: any('<' in x and '>' in x for x in l)).mean()
meta['title'] = meta.title.apply(clean_text)
meta.loc[meta.title.str.match(JUNK_TITLE_RE), 'title'] = ''
meta['features'] = meta.features.apply(clean_list)
meta['description'] = meta.description.apply(clean_list)
meta['features_text'] = meta.features.apply(lambda l: ' ; '.join(l))
meta['description_text'] = meta.description.apply(lambda l: truncate_words(' '.join(l), CFG['text']['max_desc_words']))
after_html = meta.description_text.str.contains(r'<[^>]+>', regex=True).mean()
print(f'Descriptions containing HTML tags: {before_html:.2%} before -> {after_html:.2%} after')

### B3. Brand and category normalisation
- `brand`: from `store`; if empty, from `details["Brand"]`. Patterns like "Visit the X Store" become "X".
- `brand_key`: lower-case, punctuation-free key for grouping (e.g. "L'Oréal Paris" and "L'Oreal Paris").
- `cat_path`: category list joined with " > "; `cat_l1`, `cat_l2`, `cat_l3`: levels 1–3.

In [ ]:
import unicodedata
VISIT_RE = re.compile(r'^visit the (.+?) store$', re.I)

def brand_from(row):
    b = clean_text(row['store'])
    if not b:
        try:
            b = clean_text(json.loads(row['details'] if isinstance(row['details'], str) else '{}').get('Brand', ''))
        except (ValueError, AttributeError):
            b = ''
    m = VISIT_RE.match(b)
    return m.group(1).strip() if m else b

def brand_key(b):
    b = unicodedata.normalize('NFKD', b).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', ' ', b.lower()).strip()

meta['brand'] = meta.apply(brand_from, axis=1)
meta['brand_key'] = meta.brand.apply(brand_key)
meta['categories'] = meta.categories.apply(lambda l: [clean_text(c) for c in l if clean_text(c)])
meta['cat_path'] = meta.categories.apply(lambda l: ' > '.join(l))
for lvl in (1, 2, 3):
    meta[f'cat_l{lvl}'] = meta.categories.apply(lambda l, lvl=lvl: l[lvl] if len(l) > lvl else None)

print('Items with brand:', f'{(meta.brand != "").mean():.1%}', '| distinct brand keys:', meta.brand_key.nunique())
print('Items with level-1 category:', f'{meta.cat_l1.notna().mean():.1%}')

### B4. Price
Prices arrive as strings ("$12.99", "12.99", "None", ranges). We keep the first number; anything else becomes NaN.

In [ ]:
PRICE_RE = re.compile(r'(\d+(?:[.,]\d{1,2})?)')
def parse_price(p):
    if is_missing(p):
        return np.nan
    m = PRICE_RE.search(str(p).replace(',', ''))
    return float(m.group(1)) if m else np.nan

meta['price_num'] = meta.price.apply(parse_price)
print(f'Items with numeric price: {meta.price_num.notna().mean():.1%}; median price: {meta.price_num.median():.2f}')

## Part C · Shared item set
Items with **no metadata record or an empty title** cannot get a semantic embedding. If Model 1 dropped them but
Model 0 kept them, the models would be ranking different catalogs and could not be compared. So we drop them for
everyone (`drop_items_without_title = True`). The impact was measured in notebook 02.

In [ ]:
meta['has_title'] = meta.title.str.len() > 0
valid_items = set(meta.loc[meta.has_title, 'parent_asin'])
n0 = len(ratings)
if CFG['drop_items_without_title']:
    ratings = ratings[ratings.parent_asin.isin(valid_items)]
log_step('drop interactions on items without title/metadata', n0, len(ratings),
         'enabled' if CFG['drop_items_without_title'] else 'disabled')

# Note: this can push some users/items below 5 interactions; notebook 04 re-applies k-core after subsetting.

## Save

In [ ]:
ratings_out = ratings[['user_id', 'parent_asin', 'rating', 'timestamp', 'row_id']].sort_values('row_id').reset_index(drop=True)
meta_cols = ['parent_asin', 'title', 'brand', 'brand_key', 'main_category', 'categories', 'cat_path', 'cat_l1',
             'cat_l2', 'cat_l3', 'features', 'features_text', 'description_text', 'price_num', 'average_rating',
             'rating_number', 'details', 'has_title']
meta_out = meta[meta_cols].reset_index(drop=True)

ratings_out.to_parquet(INTERIM / 'ratings_clean.parquet', index=False)
meta_out.to_parquet(INTERIM / 'meta_clean.parquet', index=False)
save_json({'steps': LOG,
           'final': {'interactions': len(ratings_out), 'users': int(ratings_out.user_id.nunique()),
                     'items': int(ratings_out.parent_asin.nunique()), 'metadata_rows': len(meta_out)}},
          INTERIM / 'cleaning_log.json')
display(pd.DataFrame(LOG))

### Examples after cleaning

In [ ]:
meta_out.loc[meta_out.has_title, ['parent_asin', 'title', 'brand', 'cat_path', 'price_num', 'description_text']].sample(5, random_state=CFG['seed'])

## Summary
All decisions and row counts are in `interim/cleaning_log.json`. These are the numbers to quote in the report's
data section.

**Next:** `04_subset_kcore_reindex.ipynb`